# Bianor Demo Setup

This notebook demonstrates the capabilities of Bianor through a local workflow.

Objectives:
- **Organized Data:** Create a local collection of the Iris dataset:
    - the `iris_features.csv` stores feature data for all observations
    - sources named as `iris_historical_**.csv` store historical patterns (e.g., `iris_historical_class_0.csv` stores sample IDs related to `Class 0` in the training set).
    - sources named as `iris_holdout_**.csv` store new observations and their class (true or predicted), (e.g., `iris_holdout_ground_truth.csv` stores sample ID / true class pairs in the holdout set).
- **Data Representation:** Fit a Self-Organizing Map (SOM) representation on the training set features.
- **Bianor Configuration:** Link the data layers and representations to the Bianor application through a `yaml` configuration.
- **Launch Bianor**

In [ ]:
from pathlib import Path
cwd = Path.cwd().resolve()

CATALOG_ROOT = cwd.parent
DATA = CATALOG_ROOT / "data"
MODELS = CATALOG_ROOT / "models"
DATA.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

In [ ]:
RANDOM_SEED = 42
VERBOSE = True

## Organized Datasets

In [ ]:
# Load the Iris dataset
from sklearn.datasets import load_iris
df_iris, target = load_iris(return_X_y=True, as_frame=True)

In [ ]:
# Export feature store
df_iris.rename_axis("id").to_csv(DATA / "iris_features.csv", index=True, header=True)

In [ ]:
# Separate holdout set
from sklearn.model_selection import train_test_split
df_train, df_holdout, target_train, target_holdout = train_test_split(df_iris, target, test_size=0.2, stratify=target, random_state=RANDOM_SEED)

In [ ]:
# Export `Class 0` (as labeled Projection Layer)
df_train[target_train == 0].rename_axis("id").index.to_frame(index=False).to_csv(DATA / "iris_historical_class_0.csv", index=False, header=True)

# Export `Class 1 & 2` (as unlabeled Projection Layer)
df_train[target_train > 0].rename_axis("id").index.to_frame(index=False).to_csv(DATA / "iris_historical_unlabeled.csv", index=False, header=True)

In [ ]:
# Export holdout ground truth ("as Predictions")
target_holdout.rename_axis("id").to_csv(DATA / "iris_holdout_ground_truth.csv", index=True, header=True)


## Modeling a Representation

Fit a `SomRepresentation` model  (a _Self-Organizing Map (SOM)_ representation) on the training dataset features.

In [ ]:
# Preprocessing: Standardize the features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train)

In [ ]:
from minisom_representation import SomRepresentation, calc_som_hyparams, plot_som_convergence_over_epochs

In [ ]:
initial_sigma_factor = 3.0
recommended_params = calc_som_hyparams(X_train_scaled, initial_sigma_factor=initial_sigma_factor, verbose=VERBOSE)
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))

In [ ]:
epoch = None # <-- find an appropriate epoch candidate
fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
    SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False),
    X_train_scaled,
    fit_type="online", te_ceiling=.1,
    epoch_step_from=2, epoch_step_to=50, epoch_step=1,
    figsize=(16, 5), show_fig=True, verbose=VERBOSE
)
print(f"\nQE (first -> last): \t {errors_qe[0]:.4f} -> {errors_qe[-1]:.4f}")
print(f"TE (first -> last): \t {errors_te[0]:.4f} -> {errors_te[-1]:.4f}")

In [ ]:
epoch = 49
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE) \
    .fit_online(X_train_scaled, num_iteration=epoch)


## Link to Bianor Application

Save the model and ensure your local Bianor catalog includes the desired entities.

In [ ]:
import shutil
import mlflow
from pathlib import Path
from mlflow.pyfunc import PythonModel
from mlflow.models import infer_signature

In [ ]:
model_name = "iris-som"
model_path = MODELS / model_name

In [ ]:
class MLflowSomModelWrapper(PythonModel):
    def __init__(self, model: SomRepresentation, scaler):
        self.model = model
        self.scaler = scaler

In [ ]:
out = Path(model_path)
if out.exists():
    shutil.rmtree(out)

In [ ]:
model_wrapped = MLflowSomModelWrapper(som_rep, scaler)
signature = infer_signature(df_train[:1])

mlflow.pyfunc.save_model(
    path=str(out),
    python_model=model_wrapped,
    signature=signature,
    pip_requirements=[
        "scikit-learn",
        "minisom-representation",
    ],
)

```yaml
# bianor_local_catalog/catalog.yaml

representations:
  - id: 1
    name: Iris Representation
    som_model_location: models/iris-som
    features_location: data/iris_features.csv

projection_layers:
  - id: 1
    name: Iris Setosa (Historical, Class 0)
    samples_location: data/iris_historical_class_0.csv
    marker_dict: '{"symbol": "circle", "size": 12, "color": "crimson"}'
    representation_id: 1
  - id: 2
    name: Iris (Historical, Unlabeled)
    samples_location: data/iris_historical_unlabeled.csv
    marker_dict: '{"symbol": "circle", "size": 12, "color": "white"}'
    representation_id: 1

predictions:
  - id: 1
    name: Iris (Holdout, Ground Truth)
    prediction_location: data/iris_holdout_ground_truth.csv
    representation_id: 1
```

## Next Steps

Install dependencies, create environment, and launch the application. Visit your Bianor instance at `http://localhost:8050`.

```bash
pip install -r requirements.txt
```

```bash
cp .env.local.example .env
```

```bash
python app.py
```